In [ ]:
import os
import pandas as pd

input_path = '../dataset/input-data.csv'
df = pd.read_csv(input_path)

In [ ]:
import torch
import pandas as pd

def generate_summaries(tokenizer, model, df, out_path, is_gpt=False, bias=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    ids = []
    summaries = [[] for _ in range(3)]  # generate 3 summaries per article

    for idx, row in df.iterrows():
        id, text = row.id, row.text

        print(f'{idx}: {id}')

        prompt = f'Summarize this article{" with " + bias + " bias" if bias else ""}: {text}'
        end_prompt = '\nSummary:'

        offset = 0

        if hasattr(tokenizer, 'model_max_length'):
            offset = 512 if is_gpt else 0
            end_prompt_len = tokenizer(end_prompt, return_tensors="pt").input_ids.shape[1]
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=tokenizer.model_max_length-offset-end_prompt_len)
            offset = inputs.input_ids.shape[1] if is_gpt else 0
        else:
            prompt += end_prompt
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

        inputs = inputs.to(device)

        summary_ids = model.generate(
            inputs.input_ids,
            min_length=10,
            max_new_tokens=512,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=len(summaries),
            pad_token_id=tokenizer.eos_token_id
        )

        for i, summary_id in enumerate(summary_ids):
            summary = tokenizer.decode(summary_id[offset:], skip_special_tokens=True)
            print(summary)
            summaries[i].append(summary)

        ids.append(id)

        # save every 30 articles
        if (idx + 1) % 30 == 0:
            save_summaries(ids, summaries, out_path)

def save_summaries(ids, summaries, out_path):
    data = {"id": ids, **{f"summary{i+1}": summaries[i] for i in range(len(summaries))}}
    df = pd.DataFrame(data)
    df.to_csv(out_path, index=False)

def summarize(tokenizer, model, df, out_path, is_gpt=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt)

def summarize_with_leaning(tokenizer, model, df, out_path, bias, is_gpt=False):
    return generate_summaries(tokenizer, model, df, out_path, is_gpt, bias)

# BART

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

In [ ]:
summarize(bart_tokenizer, bart_model, df, '../dataset/bart.csv')

In [ ]:
for leaning in ['left', 'center', 'right']:
    output_path = f'../dataset/bart-{leaning}.csv'
    summarize_with_leaning(bart_tokenizer, bart_model, df, output_path, bias=leaning)

# T5

In [ ]:
from transformers import AutoTokenizer, AutoModelWithLMHead

t5_tokenizer = AutoTokenizer.from_pretrained('t5-base')
t5_model = AutoModelWithLMHead.from_pretrained('t5-base', return_dict=True)

In [ ]:
summarize(t5_tokenizer, t5_model, df, '../dataset/t5.csv')

# GPT2 and GPT Neo

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

In [ ]:
summarize(gpt2_tokenizer, gpt2_model, df, '../dataset/gpt2.csv', is_gpt=True)

In [ ]:
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

neo_model = GPTNeoForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
neo_tokenizer = GPT2Tokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")

In [ ]:
summarize(neo_model, neo_tokenizer, df, '../dataset/neo-test.csv', is_gpt=True)

# Llama 2 (TODO)

In [ ]:
from transformers import LlamaForCausalLM, LlamaTokenizer

llama_tokenizer = LlamaTokenizer.from_pretrained("/output/path")
llama_model = LlamaForCausalLM.from_pretrained("/output/path")

In [ ]:
summarize(llama_tokenizer, llama_model, df, 'llama.csv')

In [ ]:
for leaning in ['left', 'center', 'right']:
    output_path = f'../dataset/llama-{leaning}.csv'
    summarize_with_leaning(llama_tokenizer, llama_model, df, output_path, bias=leaning)